In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score
from sklearn.preprocessing import LabelEncoder
import gc
import warnings

warnings.filterwarnings('ignore')

# LOAD & MERGE
BASE_PATH = "/kaggle/input/competitions/triagegeist"
train = pd.read_csv(f"{BASE_PATH}/train.csv")
test = pd.read_csv(f"{BASE_PATH}/test.csv")
complaints = pd.read_csv(f"{BASE_PATH}/chief_complaints.csv")
history = pd.read_csv(f"{BASE_PATH}/patient_history.csv")

def merge_all(df):
    df = df.merge(complaints[['patient_id', 'chief_complaint_raw']], on='patient_id', how='left')
    df = df.merge(history, on='patient_id', how='left')
    return df

train, test = merge_all(train), merge_all(test)

# ADVANCED CLINICAL FEATURE ENGINEERING
def engineer_clinical_logic(df):
    # Sepsis/SIRS Risk (Hand-engineered Clinical Red Flags)
    df['is_tachycardic'] = (df['heart_rate'] > 100).astype(int)
    df['is_febrile'] = (df['temperature_c'] > 38).astype(int)
    df['is_hypotensive'] = (df['systolic_bp'] < 90).astype(int)
    df['sepsis_suspected'] = ((df['is_tachycardic'] + df['is_febrile'] + (df['respiratory_rate'] > 20)) >= 2).astype(int)
    
    # Missingness as a Signal
    vitals = ['systolic_bp', 'heart_rate', 'respiratory_rate', 'temperature_c', 'spo2']
    for v in vitals:
        df[f'{v}_missing'] = df[v].isna().astype(int)
    
    # Categorical Handling for LightGBM
    cat_cols = ['arrival_mode', 'arrival_day', 'sex', 'arrival_season', 'shift', 'age_group', 'insurance_type', 'transport_origin', 'pain_location', 'mental_status_triage', 'language']
    for col in cat_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
    
    return df, cat_cols

train, cat_features = engineer_clinical_logic(train)
test, _ = engineer_clinical_logic(test)

# TEXT SEMANTICS (TF-IDF + SVD)
train['chief_complaint_raw'] = train['chief_complaint_raw'].fillna('unspecified')
test['chief_complaint_raw'] = test['chief_complaint_raw'].fillna('unspecified')
all_text = pd.concat([train['chief_complaint_raw'], test['chief_complaint_raw']])

tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1, 3), max_features=10000)
svd = TruncatedSVD(n_components=30, random_state=42)
text_vectors = svd.fit_transform(tfidf.fit_transform(all_text))

text_df = pd.DataFrame(text_vectors, columns=[f'svd_{i}' for i in range(30)])
train = pd.concat([train.reset_index(drop=True), text_df.iloc[:len(train)].reset_index(drop=True)], axis=1)
test = pd.concat([test.reset_index(drop=True), text_df.iloc[len(train):].reset_index(drop=True)], axis=1)

# ENSEMBLE TRAINING (LGBM + CATBOOST)
drop_cols = ['patient_id', 'triage_acuity', 'chief_complaint_raw', 'disposition', 'ed_los_hours']
features = [f for f in train.columns if f not in drop_cols and train[f].dtype != 'object']

X, y = train[features], train['triage_acuity'] - 1
X_test = test[features]

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lgbm_oof, cb_oof = np.zeros((len(train), 5)), np.zeros((len(train), 5))
lgbm_test, cb_test = np.zeros((len(test), 5)), np.zeros((len(test), 5))

for fold, (trn_idx, val_idx) in enumerate(folds.split(X, y)):
    print(f"--- Processing Fold {fold+1} ---")
    X_tr, X_val = X.iloc[trn_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[trn_idx], y.iloc[val_idx]
    
    # Model 1: LightGBM
    m_lgb = lgb.LGBMClassifier(n_estimators=1000, learning_rate=0.05, device='gpu', verbose=-1)
    m_lgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(100)])
    lgbm_oof[val_idx] = m_lgb.predict_proba(X_val)
    lgbm_test += m_lgb.predict_proba(X_test) / 5
    
    # Model 2: CatBoost
    m_cb = CatBoostClassifier(iterations=1000, learning_rate=0.05, task_type="GPU", verbose=0)
    m_cb.fit(X_tr, y_tr, eval_set=(X_val, y_val), early_stopping_rounds=100)
    cb_oof[val_idx] = m_cb.predict_proba(X_val)
    cb_test += m_cb.predict_proba(X_test) / 5

# BLENDING & FINAL SCORE
# Weights: 60% LGBM, 40% CatBoost for stability
final_oof_probs = (0.6 * lgbm_oof) + (0.4 * cb_oof)
final_oof_labels = np.argmax(final_oof_probs, axis=1)
qwk = cohen_kappa_score(y, final_oof_labels, weights='quadratic')
print(f"\nEnsemble Quadratic Weighted Kappa: {qwk:.5f}")

# BIAS & FAIRNESS AUDIT
print("\n--- Clinical Fairness Audit ---")
train['preds'] = final_oof_labels
for gender in train['sex'].unique():
    subset = train[train['sex'] == gender]
    score = cohen_kappa_score(subset['triage_acuity']-1, subset['preds'], weights='quadratic')
    gender_name = "Male" if gender == 0 else "Female" # Adjust based on LabelEncoder order
    print(f"QWK for Gender {gender}: {score:.4f}")

# SUBMISSION
final_test_labels = np.argmax((0.6 * lgbm_test) + (0.4 * cb_test), axis=1) + 1
submission = pd.DataFrame({'patient_id': test['patient_id'], 'triage_acuity': final_test_labels})
submission.to_csv('submission.csv', index=False)
print("\n'submission.csv' created.")